# DISMIR Fine-Tuning on `Data/training_data`

This notebook uses the DISMIR implementation in `methyldl.modelling.classifiers.dismir` and the configuration conventions from `App/main.py`.

It is split into two separate training parts because the repository contains two distinct supervision schemes:

1. hard labels with an explicit rejection class in `TrainingDataWithRejection_hg38_mincpg_4_minlen_10`
2. soft labels in `SoftLabelsTrainingData_hg38_mincpg_4_minlen_10`

Both parts:

- inspect the raw parquet inputs under `Data/training_data`
- stage a DISMIR-compatible copy with `seq` renamed to `input_ids` and `pattern` renamed to `methylation_ids`
- train DISMIR with the DMR-attention classification head using `dmr_label` as contextual input
- save an app-style YAML config for reproducibility
- summarize the resulting checkpoints and metrics

The current `DismirMLflowExperiment` wrapper is a useful reference for directory layout and logging, but it does not expose `num_labels`, `soft_labels`, or the DMR-attention setup required here. For that reason, the actual training cells below instantiate `Dismir` directly so both schemes are handled correctly.

In [1]:
import os
import sys
import json
import shutil
from pathlib import Path

import numpy as np
import torch
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd
import yaml
from IPython.display import display


repo_root = Path.cwd().parent

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from methyldl.modelling.classifiers.dismir import Dismir
from App.main import setup_logging, validate_config

%load_ext autoreload
%autoreload 2

In [2]:
raw_data_root = repo_root / "Data" / "training_data"
notebook_output_root = repo_root / "output" / "notebooks" / "dismir_fine_tuning"
prepared_data_root = notebook_output_root / "prepared_training_data"
training_output_root = notebook_output_root / "training_runs"
config_path = notebook_output_root / "dismir_fine_tuning_notebook.yaml"
log_dir = repo_root / "test_container_tmp"

for path in (notebook_output_root, prepared_data_root, training_output_root, log_dir):
    path.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {repo_root}")
print(f"Raw training data: {raw_data_root}")
print(f"Notebook output root: {notebook_output_root}")

Repository root: /home/nathan/Documents/Scolaire/7_Cesure/2_KU_Leuven/methyldl
Raw training data: /home/nathan/Documents/Scolaire/7_Cesure/2_KU_Leuven/methyldl/Data/training_data
Notebook output root: /home/nathan/Documents/Scolaire/7_Cesure/2_KU_Leuven/methyldl/output/notebooks/dismir_fine_tuning


## 1. Shared Setup

The raw parquet files currently store DISMIR inputs as `seq` and `pattern`, while `Dismir` expects `input_ids` and `methylation_ids` by default.

This notebook stages a copy of each selected dataset under `output/notebooks/dismir_fine_tuning/prepared_training_data` and applies the required column renames there. The source data under `Data/training_data` is not modified.

Training uses the DMR-attention classification head, so the staged parquet files must also preserve the `dmr_label` column for both fitting and prediction.

For the soft-label dataset, the staging helper also validates `soft_label` vectors and can drop rows whose probability mass is zero.

In [3]:
available_datasets = sorted(path.name for path in raw_data_root.iterdir() if path.is_dir())
print("Available datasets:")
for name in available_datasets:
    print(f"- {name}")

SCHEMES = {
    "hard_labels": {
        "dataset_name": "TrainingDataWithRejection_hg38_mincpg_4_minlen_10",
        "use_soft_labels": False,
        "description": "Hard labels with an explicit rejection class",
    },
    "soft_labels": {
        "dataset_name": "SoftLabelsTrainingData_hg38_mincpg_4_minlen_10",
        "use_soft_labels": True,
        "description": "Soft-label supervision using probability vectors",
    },
}
MAX_SEQUENCE_LENGTH = 150
MODEL_FLAVOR = "minigru"
CLASSIFIER_TYPE = "dmr_attention_based"
DMR_LABEL_COLUMN = "dmr_label"
COMMON_TRAINING_PARAMS = {
    "epochs": 10,
    "batch_size": 128,
    "patience": 3,
    "optimizer_type": "Adam",
    "lr": 0.001,
    "momentum": 0.9,
    "weight_decay": 0.000001,
}

required = [config["dataset_name"] for config in SCHEMES.values()]
missing = [name for name in required if name not in available_datasets]
if missing:
    raise ValueError(f"Missing required dataset(s): {missing}")

scheme_defaults = {
    scheme_name: f"{config['dataset_name']} ({config['description']})"
    for scheme_name, config in SCHEMES.items()
}

print("\nNotebook defaults:")
display(
    pd.Series(
        {
            **scheme_defaults,
            "max_sequence_length": MAX_SEQUENCE_LENGTH,
            "model_flavor": MODEL_FLAVOR,
            "classifier_type": CLASSIFIER_TYPE,
            "dmr_label_column": DMR_LABEL_COLUMN,
            **COMMON_TRAINING_PARAMS,
        },
        name="value",
    )
)

Available datasets:
- SoftLabelsTrainingData_hg38_mincpg_4_minlen_10
- TrainingDataWithRejection_hg38_mincpg_4_minlen_10

Notebook defaults:


hard_labels            TrainingDataWithRejection_hg38_mincpg_4_minlen...
soft_labels            SoftLabelsTrainingData_hg38_mincpg_4_minlen_10...
max_sequence_length                                                  150
model_flavor                                                     minigru
classifier_type                                      dmr_attention_based
dmr_label_column                                               dmr_label
epochs                                                                10
batch_size                                                           128
patience                                                               3
optimizer_type                                                      Adam
lr                                                                 0.001
momentum                                                             0.9
weight_decay                                                    0.000001
Name: value, dtype: object

## 2. Shared Helpers

These helpers stage raw parquet files into the DISMIR input schema, write notebook-specific YAML configs, train a model, and collect split-level summaries for later inspection.

In [4]:
SPLITS = ("train", "valid", "test")
SCHEME_OUTPUT_ROOT = notebook_output_root / "schemes"
SCHEME_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

In [5]:
def summarize_raw_dataset(dataset_name: str) -> pd.DataFrame:
    """Summarize the raw parquet files for one training dataset.

    The summary is used to confirm the supervision scheme before training,
    including whether a soft-label vector is present and how many label values
    appear in each split.
    """
    rows = []
    dataset_dir = raw_data_root / dataset_name
    for split in SPLITS:
        split_path = dataset_dir / f"{split}.parquet"
        df = pd.read_parquet(split_path)
        row = {
            "dataset": dataset_name,
            "split": split,
            "rows": len(df),
            "num_label_values": int(df["label"].nunique()),
            "max_label": int(df["label"].max()),
            "has_soft_label": bool("soft_label" in df.columns),
        }
        if "soft_label" in df.columns:
            first_valid = next(
                (value for value in df["soft_label"] if value is not None and len(value) > 0),
                None,
            )
            row["soft_label_dim"] = None if first_valid is None else int(len(first_valid))
            row["zero_mass_soft_labels"] = int(
                df["soft_label"].apply(lambda x: float(np.sum(x)) <= 0.0).sum()
            )
        rows.append(row)
    return pd.DataFrame(rows)

In [6]:
def normalize_soft_label(value) -> list[float]:
    """Normalize a soft-label vector and return it as a Python list.

    An empty list is returned for zero-mass vectors so the staging step can
    filter them out explicitly instead of silently keeping invalid labels.
    """
    array = np.asarray(value, dtype=np.float32)
    total = float(array.sum())
    if total <= 0.0:
        return []
    return (array / total).tolist()


def infer_num_labels(dataset_dir: Path, use_soft_labels: bool = False) -> int:
    """Infer the classifier output dimension from the staged training data."""
    train_df = pd.read_parquet(dataset_dir / "train.parquet")
    if use_soft_labels:
        first_soft_label = next(
            value for value in train_df["soft_label"] if value is not None and len(value) > 0
        )
        return int(len(first_soft_label))
    return int(train_df["label"].max()) + 1


def infer_num_dmr_labels(dataset_dir: Path, dmr_label_col: str = DMR_LABEL_COLUMN) -> int:
    """Infer the number of distinct DMR labels from the staged training data."""
    train_df = pd.read_parquet(dataset_dir / "train.parquet", columns=[dmr_label_col])
    return int(train_df[dmr_label_col].max()) + 1

In [7]:
def build_notebook_config(
    dataset_dir: Path,
    scheme_name: str,
    num_labels: int,
    num_dmr_labels: int,
    use_soft_labels: bool = False,
) -> dict:
    """Build a notebook-local training configuration dictionary.

    The structure mirrors the app configuration format so the staged run is easy
    to compare with command-line training runs.
    """
    return {
        "scheme_name": scheme_name,
        "model": {
            "architecture": "dismir",
            "flavor": MODEL_FLAVOR,
            "classifier_type": CLASSIFIER_TYPE,
            "dmr_label_col": DMR_LABEL_COLUMN,
            "num_labels": num_labels,
            "num_dmr_labels": num_dmr_labels,
            "soft_labels": use_soft_labels,
        },
        "data_path": str(dataset_dir.parent.relative_to(repo_root)),
        "datasets": [dataset_dir.name],
        "max_sequence_length": MAX_SEQUENCE_LENGTH,
        "splits": list(SPLITS),
        "training": {
            **COMMON_TRAINING_PARAMS,
            "output_dir": str((SCHEME_OUTPUT_ROOT / scheme_name / "artifacts").relative_to(repo_root)),
        },
        "mlflow": {
            "experiment_name": f"dismir_notebook_{scheme_name}",
            "tracking_uri": None,
        },
    }


def write_notebook_config(config: dict, scheme_name: str) -> Path:
    """Write a YAML snapshot of the notebook configuration to disk."""
    scheme_dir = SCHEME_OUTPUT_ROOT / scheme_name
    scheme_dir.mkdir(parents=True, exist_ok=True)
    scheme_config_path = scheme_dir / "config.yaml"
    with scheme_config_path.open("w", encoding="utf-8") as handle:
        yaml.safe_dump(config, handle, sort_keys=False)
    return scheme_config_path


def collect_split_metrics(
    model: Dismir,
    dataset_dir: Path,
    use_soft_labels: bool = False,
) -> pd.DataFrame:
    """Collect split-level metrics from a trained DISMIR model.

    For the soft-label scheme, the notebook reports classification metrics against
    the argmax class of each probability vector so the results remain comparable
    to the hard-label run.
    """
    rows = []
    for split in SPLITS:
        df = pd.read_parquet(dataset_dir / f"{split}.parquet")
        _, predicted_labels = model.predict(
            df["input_ids"].tolist(),
            df["methylation_ids"].tolist(),
            dmr_ids=df[DMR_LABEL_COLUMN].to_numpy(),
            batch_size=COMMON_TRAINING_PARAMS["batch_size"],
        )
        if use_soft_labels:
            # The training target is a probability vector, but the summary table
            # needs a single reference class for accuracy and F1.
            true_labels = np.vstack(df["soft_label"].to_numpy()).argmax(axis=1)
        else:
            true_labels = df["label"].to_numpy()

        if split in {"valid", "test"}:
            loss, reported_accuracy = model.evaluate(split=split)
            loss = float(loss)
            reported_accuracy = float(reported_accuracy)
        else:
            # The DISMIR helper only exposes evaluation for valid/test, so the
            # training split uses prediction-based metrics only.
            loss = float("nan")
            reported_accuracy = float(accuracy_score(true_labels, predicted_labels))

        rows.append(
            {
                "split": split,
                "rows": len(df),
                "loss": loss,
                "accuracy": reported_accuracy,
                "weighted_f1": float(f1_score(true_labels, predicted_labels, average="weighted", zero_division=0)),
                "simple_accuracy": float(accuracy_score(true_labels, predicted_labels)),
                "num_predicted_classes": int(np.unique(predicted_labels).size),
            }
        )
    return pd.DataFrame(rows)


In [8]:
def stage_dataset_for_dismir(
    scheme_name: str,
    drop_zero_mass_soft_labels: bool = True,
) -> tuple[Path, pd.DataFrame]:
    """Create or reuse a DISMIR-compatible staged copy for one scheme.

    The scheme definition is the source of truth for both the underlying dataset
    and whether the staged data should be treated as soft-label supervision.
    """
    scheme_config = SCHEMES[scheme_name]
    dataset_name = scheme_config["dataset_name"]
    use_soft_labels = scheme_config["use_soft_labels"]

    source_dir = raw_data_root / dataset_name
    staged_dir = prepared_data_root / scheme_name / dataset_name
    staged_split_paths = [staged_dir / f"{split}.parquet" for split in SPLITS]

    if staged_dir.exists() and all(path.exists() for path in staged_split_paths):
        staging_summary = []
        for split, staged_path in zip(SPLITS, staged_split_paths):
            raw_df = pd.read_parquet(source_dir / f"{split}.parquet", columns=["label"])
            staged_df = pd.read_parquet(staged_path)
            # When reusing cached staged data, infer how many rows were removed by
            # comparing raw and staged split sizes instead of recomputing the staging step.
            dropped_rows = len(raw_df) - len(staged_df) if use_soft_labels else 0
            staging_summary.append(
                {
                    "split": split,
                    "rows": len(staged_df),
                    "dropped_rows": dropped_rows,
                    "columns": ", ".join(staged_df.columns[:8]) + (" ..." if len(staged_df.columns) > 8 else ""),
                }
            )
        return staged_dir, pd.DataFrame(staging_summary)

    if staged_dir.exists():
        shutil.rmtree(staged_dir)
    staged_dir.mkdir(parents=True, exist_ok=True)

    staging_summary = []
    for split in SPLITS:
        df = pd.read_parquet(source_dir / f"{split}.parquet").copy()
        # DISMIR expects input_ids/methylation_ids, while the raw training data
        # is stored as seq/pattern.
        df.rename(columns={"seq": "input_ids", "pattern": "methylation_ids"}, inplace=True)

        dropped_rows = 0
        if use_soft_labels:
            if "soft_label" not in df.columns:
                raise ValueError(f"Dataset {dataset_name} does not contain a soft_label column.")
            normalized = df["soft_label"].apply(normalize_soft_label)
            if drop_zero_mass_soft_labels:
                # Zero-length normalized labels correspond to invalid zero-mass
                # vectors, so we remove them before saving the staged parquet.
                valid_mask = normalized.apply(len) > 0
                dropped_rows = int((~valid_mask).sum())
                df = df.loc[valid_mask].copy()
                normalized = normalized.loc[valid_mask]
            df["soft_label"] = normalized.tolist()

        df.to_parquet(staged_dir / f"{split}.parquet", index=False)
        staging_summary.append(
            {
                "split": split,
                "rows": len(df),
                "dropped_rows": dropped_rows,
                "columns": ", ".join(df.columns[:8]) + (" ..." if len(df.columns) > 8 else ""),
            }
        )

    return staged_dir, pd.DataFrame(staging_summary)

In [9]:
def train_dismir_scheme(scheme_name: str) -> dict:
    """Stage data, train DISMIR, and persist run artifacts for one scheme."""
    scheme_config = SCHEMES[scheme_name]
    dataset_name = scheme_config["dataset_name"]
    use_soft_labels = scheme_config["use_soft_labels"]

    # Prepare the training data associated with this scheme.
    staged_dir, staging_summary = stage_dataset_for_dismir(
        scheme_name=scheme_name,
        drop_zero_mass_soft_labels=use_soft_labels,
    )
    # Compute the number of label classes and DMR contexts after staging.
    num_labels = infer_num_labels(staged_dir, use_soft_labels=use_soft_labels)
    num_dmr_labels = infer_num_dmr_labels(staged_dir)

    # Build and validate a config snapshot for this notebook run.
    config = build_notebook_config(
        dataset_dir=staged_dir,
        scheme_name=scheme_name,
        num_labels=num_labels,
        num_dmr_labels=num_dmr_labels,
        use_soft_labels=use_soft_labels,
    )
    config_path = write_notebook_config(config, scheme_name)
    validate_config(config, "fine_tune")

    # Set up output directories.
    scheme_dir = SCHEME_OUTPUT_ROOT / scheme_name
    artifacts_dir = scheme_dir / "artifacts"
    weights_dir = scheme_dir / "weights"
    artifacts_dir.mkdir(parents=True, exist_ok=True)
    weights_dir.mkdir(parents=True, exist_ok=True)

    logger = setup_logging(verbose=False)
    logger.info("Starting notebook training for scheme=%s dataset=%s", scheme_name, dataset_name)

    # Instantiate DISMIR with the DMR-attention classification head.
    model = Dismir(
        max_sequence_length=MAX_SEQUENCE_LENGTH,
        train_data_path=str(staged_dir / "train.parquet"),
        test_data_path=str(staged_dir / "test.parquet"),
        valid_data_path=str(staged_dir / "valid.parquet"),
        flavour=MODEL_FLAVOR,
        num_labels=num_labels,
        classifier_type=CLASSIFIER_TYPE,
        num_dmr_labels=num_dmr_labels,
        dmr_label_col=DMR_LABEL_COLUMN,
        soft_labels=use_soft_labels,
    )
    print(f"Device used for training: {model.device}")

    model.train(train_dir=str(weights_dir), verbose=1, **COMMON_TRAINING_PARAMS)

    best_weights_path = weights_dir / "weight.pt"
    if best_weights_path.exists():
        # Reload the best checkpoint saved by early stopping before computing the
        # final summary tables and writing artifact metadata.
        model.model.load_state_dict(torch.load(best_weights_path, map_location=model.device))

    metrics_df = collect_split_metrics(model, staged_dir, use_soft_labels=use_soft_labels)
    history_df = pd.DataFrame(model.history)
    metrics_path = scheme_dir / "metrics.csv"
    history_path = scheme_dir / "history.csv"
    metrics_df.to_csv(metrics_path, index=False)
    history_df.to_csv(history_path, index=False)

    summary = {
        "scheme_name": scheme_name,
        "dataset_name": dataset_name,
        "classifier_type": CLASSIFIER_TYPE,
        "dmr_label_col": DMR_LABEL_COLUMN,
        "use_soft_labels": use_soft_labels,
        "num_labels": num_labels,
        "num_dmr_labels": num_dmr_labels,
        "staged_dataset_dir": str(staged_dir),
        "config_path": str(config_path),
        "weights_path": str(best_weights_path),
        "metrics_path": str(metrics_path),
        "history_path": str(history_path),
    }
    with (scheme_dir / "summary.json").open("w", encoding="utf-8") as handle:
        json.dump(summary, handle, indent=2)

    return {
        "model": model,
        "config": config,
        "config_path": config_path,
        "staged_dir": staged_dir,
        "staging_summary": staging_summary,
        "history_df": history_df,
        "metrics_df": metrics_df,
        "summary": summary,
    }

## 3. Inspect the Two Training Schemes

This section confirms the label structure before launching training with the DMR-attention head:

- the hard-label scheme includes an explicit rejection label
- the soft-label scheme includes a `soft_label` probability vector in addition to the hard `label` column
- both schemes provide `dmr_label`, which is used as the contextual input for the attention-based classifier

In [10]:
raw_dataset_summary = pd.concat(
    [
        summarize_raw_dataset(config["dataset_name"]).assign(scheme=scheme_name)
        for scheme_name, config in SCHEMES.items()
    ],
    ignore_index=True,
)
display(raw_dataset_summary[["scheme", "dataset", "split", "rows", "num_label_values", "max_label", "has_soft_label", "soft_label_dim", "zero_mass_soft_labels"]])

soft_train_preview = pd.read_parquet(
    raw_data_root / SCHEMES["soft_labels"]["dataset_name"] / "train.parquet",
    columns=["label", "soft_label"],
).head(3)
soft_train_preview = soft_train_preview.assign(
    soft_label_dimension=soft_train_preview["soft_label"].apply(len),
    soft_label_mass=soft_train_preview["soft_label"].apply(lambda values: float(np.sum(values))),
)
display(soft_train_preview[["label", "soft_label_dimension", "soft_label_mass"]])

,scheme,dataset,split,rows,num_label_values,max_label,has_soft_label,soft_label_dim,zero_mass_soft_labels
0,hard_labels,TrainingDataWithRejection_hg38_mincpg_4_minlen_10,train,132485,40,39,False,NaN,NaN
1,hard_labels,TrainingDataWithRejection_hg38_mincpg_4_minlen_10,valid,59274,39,39,False,NaN,NaN
2,hard_labels,TrainingDataWithRejection_hg38_mincpg_4_minlen_10,test,4445834,39,39,False,NaN,NaN
3,soft_labels,SoftLabelsTrainingData_hg38_mincpg_4_minlen_10,train,2820354,40,39,True,39.0,0.0
4,soft_labels,SoftLabelsTrainingData_hg38_mincpg_4_minlen_10,valid,920766,40,39,True,39.0,0.0
5,soft_labels,SoftLabelsTrainingData_hg38_mincpg_4_minlen_10,test,896473,40,39,True,39.0,0.0


,label,soft_label_dimension,soft_label_mass
0,39,39,1.0
1,39,39,1.0
2,39,39,1.0


## 4. Part A: Hard-Label Scheme

### 4.1. Hard-Label Training with vanilla parameters

This section runs the `hard_labels` scheme, which is linked to `TrainingDataWithRejection_hg38_mincpg_4_minlen_10`.

That scheme uses the integer `label` column directly, including the rejection class, and trains DISMIR with the DMR-attention classification head using `dmr_label` as contextual input.

In [ ]:
hard_label_run = train_dismir_scheme("hard_labels")

In [ ]:
print("Hard-label config:")
with open(hard_label_run["config_path"], "r", encoding="utf-8") as handle:
    print(handle.read())

print("Hard-label staging summary:")
display(hard_label_run["staging_summary"])

print("Hard-label training history tail:")
display(hard_label_run["history_df"].tail())

print("Hard-label split metrics:")
display(hard_label_run["metrics_df"])

print("Saved artifacts:")
display(pd.Series(hard_label_run["summary"], name="value"))

### 4.2. Hard-Label Training with different parameters and schedulers

In this section, we train the same `hard_labels` scheme with different hyperparameters and learning rate schedulers to see if we can improve performance.

In [11]:
SCHEMES

{'hard_labels': {'dataset_name': 'TrainingDataWithRejection_hg38_mincpg_4_minlen_10',
  'use_soft_labels': False,
  'description': 'Hard labels with an explicit rejection class'},
 'soft_labels': {'dataset_name': 'SoftLabelsTrainingData_hg38_mincpg_4_minlen_10',
  'use_soft_labels': True,
  'description': 'Soft-label supervision using probability vectors'}}

In [ ]:
scheme_name = "hard_labels"
scheme_config = SCHEMES[scheme_name]
dataset_name = scheme_config["dataset_name"]
use_soft_labels = False

# Prepare the training data associated with this scheme.
staged_dir, staging_summary = stage_dataset_for_dismir(
    scheme_name=scheme_name,
    drop_zero_mass_soft_labels=use_soft_labels,
)
    # Compute the number of label classes and DMR contexts after staging.
    num_labels = infer_num_labels(staged_dir, use_soft_labels=use_soft_labels)
    num_dmr_labels = infer_num_dmr_labels(staged_dir)

    # Build and validate a config snapshot for this notebook run.
    config = build_notebook_config(
        dataset_dir=staged_dir,
        scheme_name=scheme_name,
        num_labels=num_labels,
        num_dmr_labels=num_dmr_labels,
        use_soft_labels=use_soft_labels,
    )
    config_path = write_notebook_config(config, scheme_name)
    validate_config(config, "fine_tune")

    # Set up output directories.
    scheme_dir = SCHEME_OUTPUT_ROOT / scheme_name
    artifacts_dir = scheme_dir / "artifacts"
    weights_dir = scheme_dir / "weights"
    artifacts_dir.mkdir(parents=True, exist_ok=True)
    weights_dir.mkdir(parents=True, exist_ok=True)

    logger = setup_logging(verbose=False)
    logger.info("Starting notebook training for scheme=%s dataset=%s", scheme_name, dataset_name)

    # Instantiate DISMIR with the DMR-attention classification head.
    model = Dismir(
        max_sequence_length=MAX_SEQUENCE_LENGTH,
        train_data_path=str(staged_dir / "train.parquet"),
        test_data_path=str(staged_dir / "test.parquet"),
        valid_data_path=str(staged_dir / "valid.parquet"),
        flavour=MODEL_FLAVOR,
        num_labels=num_labels,
        classifier_type=CLASSIFIER_TYPE,
        num_dmr_labels=num_dmr_labels,
        dmr_label_col=DMR_LABEL_COLUMN,
        soft_labels=use_soft_labels,
    )
    print(f"Device used for training: {model.device}")

    model.train(train_dir=str(weights_dir), verbose=1, **COMMON_TRAINING_PARAMS)

## 5. Part B: Soft-Label Scheme

This section runs the `soft_labels` scheme, which is linked to `SoftLabelsTrainingData_hg38_mincpg_4_minlen_10`.

For that scheme, the DISMIR target comes from the `soft_label` probability vector rather than from the hard `label` column. Training still uses the DMR-attention classification head, with `dmr_label` supplying the DMR context for each read.

In [ ]:
soft_label_run = train_dismir_scheme("soft_labels")

print("Soft-label config:")
with open(soft_label_run["config_path"], "r", encoding="utf-8") as handle:
    print(handle.read())

print("Soft-label staging summary:")
display(soft_label_run["staging_summary"])

print("Soft-label training history tail:")
display(soft_label_run["history_df"].tail())

print("Soft-label split metrics:")
display(soft_label_run["metrics_df"])

print("Saved artifacts:")
display(pd.Series(soft_label_run["summary"], name="value"))